# Historical Colab Pilot #1 — `deepseek-ai/deepseek-llm-7b-chat` (`afbda8b3`)

- **Target runtime:** Google Colab **L4 GPU (24 GB)** (or A100/H100/G4; refuses T4 because native precision is `bfloat16`).
- **Default mode (`RUN_FULL_245 = False`):** Runs only the 3-prompt native-precision preflight (`SHORT`, `LONG` with full `4,075` output allowance inside the `4,096` window, and `G1` values) + strict `phase36` fidelity assertions (`0` `\ufffd`, `\x00`, `Ġ`, `Ċ`, or role continuations; `finish_reason == "stop"`), then stops the local vLLM server in Cell 5.
- **Full cell (`RUN_FULL_245 = True`):** Collects all `125` freeflow + `120` values traces, runs the strict `245`-trace audit, writes `SHA256SUMS.txt`, and downloads a `.tar.gz` bundle.
- **When finished:** After any download completes, disconnect and delete the Colab runtime (`Runtime -> Disconnect and delete runtime`, or set `UNASSIGN_RUNTIME_NOW = True` in Cell 5) so the GPU stops burning compute units.

In [ ]:
# @title 1. Verify BF16 GPU + install pinned vLLM & transformers
import os, shutil, subprocess, torch

assert torch.cuda.is_available(), "No CUDA GPU attached — select Runtime -> Change runtime type -> L4 GPU."
gpu_name = torch.cuda.get_device_name(0)
cc_major, cc_minor = torch.cuda.get_device_capability(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
free_disk_gib = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"GPU: {gpu_name} (sm_{cc_major}{cc_minor}), VRAM: {vram_gib:.1f} GiB, Free disk: {free_disk_gib:.1f} GiB")

assert cc_major >= 8, f"Refusing {gpu_name} (sm_{cc_major}{cc_minor}): native bfloat16 requires Ampere+ (L4/A100/H100/G4), not T4."
assert vram_gib >= 21.5, f"Need >=22 GiB VRAM for 13.8 GB BF16 weights + 4k KV cache; got {vram_gib:.1f} GiB."

subprocess.run(["pip", "install", "-q", "vllm==0.34.0", "transformers==5.22.2", "tokenizers==0.23.3", "httpx"], check=True)
CORPUS_COMMIT = "6620c7d00eddaa90444b36673848cd69dc5eb37b"
repo_dir = "/content/model-personality-corpus-v2"
if not os.path.exists(repo_dir):
    os.makedirs(repo_dir, exist_ok=True)
    subprocess.run(["git", "init", repo_dir], check=True)
subprocess.run(["git", "-C", repo_dir, "fetch", "--depth", "1",
                "https://github.com/swombat/model-personality-corpus-v2.git", CORPUS_COMMIT], check=True)
subprocess.run(["git", "-C", repo_dir, "checkout", "--detach", "FETCH_HEAD"], check=True)

In [ ]:
# @title 2. Start loopback-only vLLM server at pinned revision afbda8b3 (bfloat16)
import json, secrets, subprocess, time, urllib.request
import transformers, vllm

MODEL = "deepseek-ai/deepseek-llm-7b-chat"
REVISION = "afbda8b347ec881666061fa67447046fc5164ec8"
LABEL = "deepseek-llm-7b-chat-colab-vllm-bf16-rafbda8b3"
RUNTIME_VERSION = f"vllm {vllm.__version__}; transformers {transformers.__version__}; torch {torch.__version__}"
TOKEN = secrets.token_urlsafe(24)
print("Runtime:", RUNTIME_VERSION)

server_log = open("/content/vllm.log", "w")
server_proc = subprocess.Popen([
    "vllm", "serve", MODEL,
    "--revision", REVISION,
    "--dtype", "bfloat16",
    "--max-model-len", "4096",
    "--host", "127.0.0.1",
    "--port", "8000",
    "--api-key", TOKEN,
], stdout=server_log, stderr=subprocess.STDOUT)

deadline = time.time() + 600
while time.time() < deadline:
    assert server_proc.poll() is None, open("/content/vllm.log").read()[-3000:]
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/v1/models", headers={"Authorization": f"Bearer {TOKEN}"})
        with urllib.request.urlopen(req, timeout=5) as r:
            if r.status == 200:
                print("vLLM ready:", json.load(r))
                break
    except Exception:
        time.sleep(5)
else:
    raise TimeoutError(open("/content/vllm.log").read()[-3000:])

In [ ]:
# @title 3. Run 3-prompt Pilot Preflight (SHORT + LONG at 4,075 allowance + G1 values)
import sys
sys.path.insert(0, "/content/model-personality-corpus-v2/scripts")
from colab_capture import run_preflight

# Templated prompt_tokens for deepseek-llm-7b-chat (afbda8b3): LONG = 21 tokens -> 4096 - 21 = 4075 native output allowance.
preflight_summary = run_preflight("http://127.0.0.1:8000/v1", TOKEN, MODEL, ff_max=4075, val_max=2000, stop=None)
print(json.dumps(preflight_summary, indent=2, ensure_ascii=False))

In [ ]:
# @title 4. Optional: Collect full 125 + 120 cell, run strict phase36 audit, and download .tar.gz
import hashlib, os, tarfile
from pathlib import Path
from colab_capture import audit_cell_on_disk

RUN_FULL_245 = False  # @param {type:"boolean"}

if not RUN_FULL_245:
    print("Preflight complete (RUN_FULL_245=False). Set RUN_FULL_245=True and re-run this cell to capture all 245 traces.")
else:
    repo = Path("/content/model-personality-corpus-v2")
    env = dict(os.environ)
    env.update({
        "LOCAL_OPENAI_BASE_URL": "http://127.0.0.1:8000/v1/chat/completions",
        "LOCAL_OPENAI_API_KEY": TOKEN,
        "LOCAL_MODEL_RUNTIME": "vllm-openai (colab-notebook)",
        "LOCAL_MODEL_RUNTIME_VERSION": RUNTIME_VERSION,
        "LOCAL_MODEL_REVISION": REVISION,
        "LOCAL_MODEL_WEIGHT_PRECISION": "bfloat16",
        "LOCAL_MODEL_HARDWARE": f"Google Colab 1x {gpu_name}",
    })
    t0 = time.time()
    subprocess.run(["python3", str(repo / "scripts/run_freeflow_multi.py"), "local-openai", MODEL,
                    "--label", LABEL, "--workers", "6", "--max-tokens", "4075"], env=env, check=True)
    subprocess.run(["python3", str(repo / "scripts/run_values_v2.py"), "local-openai", MODEL,
                    "--label", LABEL, "--workers", "6", "--max-tokens", "2000"], env=env, check=True)
    audit = audit_cell_on_disk(LABEL, MODEL, REVISION)
    print(f"Finished in {(time.time() - t0)/60:.1f} min. Audit:")
    print(json.dumps(audit, indent=2, ensure_ascii=False))
    assert audit["publication_ready_raw"], "Strict fidelity audit failed — inspect issues above."

    ff_dir = repo / "data/traces_freeflow" / f"freeflow_{LABEL}"
    val_dir = repo / "data/traces_values" / LABEL
    manifest_lines = []
    for p in sorted([*ff_dir.glob("*.json"), *val_dir.glob("*.json")]):
        h = hashlib.sha256(p.read_bytes()).hexdigest()
        manifest_lines.append(f"{h}  {p.relative_to(repo)}")
    sums_path = Path(f"/content/{LABEL}-SHA256SUMS.txt")
    sums_path.write_text("\n".join(manifest_lines) + "\n")

    bundle = Path(f"/content/{LABEL}.tar.gz")
    with tarfile.open(bundle, "w:gz") as tar:
        tar.add(ff_dir, arcname=f"data/traces_freeflow/freeflow_{LABEL}")
        tar.add(val_dir, arcname=f"data/traces_values/{LABEL}")
        tar.add(sums_path, arcname="SHA256SUMS.txt")
    print("Bundle ready:", bundle, f"({bundle.stat().st_size / 1024:.1f} KiB)")
    from google.colab import files
    files.download(str(bundle))

In [ ]:
# @title 5. Stop local vLLM server & release GPU runtime
UNASSIGN_RUNTIME_NOW = False  # @param {type:"boolean"}

if "server_proc" in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server_proc.kill()
    print("Stopped local vLLM server.")
else:
    print("vLLM server already stopped.")

if UNASSIGN_RUNTIME_NOW:
    from google.colab import runtime
    print("Unassigning Colab runtime now...")
    runtime.unassign()
else:
    print("IMPORTANT: Once any bundle download finishes, disconnect and delete the Colab runtime "
          "(Runtime -> Disconnect and delete runtime, or re-run this cell with UNASSIGN_RUNTIME_NOW=True) "
          "so the GPU VM does not stay attached and burn CCUs.")